# VADER vs FinBERT on Financial PhraseBank

**Author.** Jarman Perry — Honors extension, undergraduate ML / neural networks course.

**Abstract.** This notebook is a head-to-head comparison of two sentiment models on financial news sentences from the Financial PhraseBank (Malo et al., 2014): the rule-based **VADER** baseline (Hutto & Gilbert, 2014) and the Transformer-based **FinBERT** (Araci, 2019; HF model `ProsusAI/finbert`). The point isn't just to show that FinBERT wins — that's expected — but to make the architectural leap from a hand-built lexicon to attention-based contextual embeddings legible, and to surface the kinds of sentences where the gap actually shows up. Both models are evaluated zero-shot on a stratified held-out test set so the comparison is apples-to-apples.

**What you'll see below.** A reproducible setup cell, the cleaned dataset, a baseline VADER run, a FinBERT run, side-by-side metrics, four comparison figures, and an edge-case section where I read individual sentences to understand *why* one model beats the other on each.

## 1. Setup

All pipeline logic lives under `src/`. The notebook only orchestrates and renders — that way `pytest` and the notebook are exercising the same code. The setup cell seeds every RNG we touch, picks CPU or CUDA automatically, and makes sure the output directories exist.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# The notebook lives in notebooks/, so the repo root is one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import logging

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

from src import config
from src.config import set_seeds, ensure_dirs
from src.data_loader import load_and_split
from src.vader_model import run_vader
from src.finbert_model import run_finbert
from src.evaluate import (
    compute_metrics,
    write_metrics_json,
    build_predictions_frame,
    write_predictions_csv,
    extract_edge_cases,
    write_disagreements_csv,
)
from src.visualize import (
    plot_label_distribution,
    plot_confusion_matrices,
    plot_per_class_f1,
    plot_finbert_confidence_hist,
    plot_per_class_precision_recall,
)

logging.getLogger().setLevel(logging.INFO)
set_seeds(config.SEED)
ensure_dirs()

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch {torch.__version__}, device = {device}")
print(f"Repo root: {REPO_ROOT}")

## 2. Data

Financial PhraseBank, 75% inter-annotator agreement split. The loader handles latin-1 encoding and falls back to the Hugging Face mirror if the local file isn't there. Labels are mapped to `{negative: 0, neutral: 1, positive: 2}` and the cleaned dataset is persisted to `data/processed/phrasebank.parquet` so the rest of the pipeline (and a grader poking around) doesn't have to re-run the loader.

PhraseBank is heavily neutral-skewed, so we use a **stratified** 80/20 split (`random_state=42`) — random splitting could easily under-represent the negative class on the test side.

In [ ]:
split = load_and_split(use_50agree=False, test_size=config.TEST_SIZE, seed=config.SEED)
train_df, test_df = split.train, split.test

print(f"train: {len(train_df)} rows | test: {len(test_df)} rows")
print("\nTrain class counts:")
print(train_df['label_str'].value_counts())
print("\nTest class counts:")
print(test_df['label_str'].value_counts())

test_df.head(5)

In [ ]:
full_df = pd.concat([train_df, test_df], ignore_index=True)
label_dist_path = plot_label_distribution(full_df)
display(Image(filename=str(label_dist_path)))

## 3. Baseline — VADER

VADER (Hutto & Gilbert, 2014) is a hand-built sentiment model: a lexicon mapping words to valence floats, plus a small set of rules for boosters ("very"), negation ("not good"), capitalisation, punctuation emphasis, and contrastive conjunctions ("but"). It produces a single `compound` score in `[-1, 1]` per sentence; we threshold at the canonical ±0.05 to recover three classes.

**The architectural point.** VADER is bag-of-words plus rules — every word's valence is fixed at lookup time, regardless of the surrounding sentence. There is no notion of *context*. That's exactly why it struggles with financial language: words like "missed," "compressed," or "tightened" can flip polarity depending on what they're modifying, and VADER's lexicon can't represent that.

In [ ]:
vader_pred, vader_compound = run_vader(test_df['sentence'].tolist())
vader_metrics = compute_metrics(test_df['label'].to_numpy(), vader_pred)

print(f"VADER accuracy:  {vader_metrics['accuracy']:.4f}")
print(f"VADER macro-F1:  {vader_metrics['macro_f1']:.4f}")
print()
print(vader_metrics['classification_report'])

## 4. Transformer — FinBERT

FinBERT (Araci, 2019; [model card](https://huggingface.co/ProsusAI/finbert)) is BERT-base fine-tuned on the Reuters TRC2-financial corpus and on PhraseBank itself. Architecturally, it's a stack of self-attention layers that produces a *contextual* embedding for every token: the same word in two different sentences gets two different vectors. That contextuality is the leap. "Beat" in "Acme beat estimates" lives near the cluster of positive earnings verbs; "beat" in "the rain beat down on the field" doesn't. A lexicon model can't represent that distinction; an attention-based encoder can.

Implementation notes:

- Batched inference at 32, padding + truncation at 256 tokens (PhraseBank sentences are short, so the truncation almost never fires).
- We map FinBERT's native string labels back to our canonical integer scheme. The model's `id2label` is `{0: 'positive', 1: 'negative', 2: 'neutral'}` — *not* alphabetical — so blindly using `argmax` indices would silently swap negative and positive in your metrics. The `FinBertClassifier` wrapper handles that remap so we never index into raw logits.

In [ ]:
finbert_pred, finbert_conf = run_finbert(
    test_df['sentence'].tolist(),
    batch_size=config.FINBERT_BATCH_SIZE,
    max_length=config.FINBERT_MAX_LENGTH,
)
finbert_metrics = compute_metrics(test_df['label'].to_numpy(), finbert_pred)

print(f"FinBERT accuracy:  {finbert_metrics['accuracy']:.4f}")
print(f"FinBERT macro-F1:  {finbert_metrics['macro_f1']:.4f}")
print()
print(finbert_metrics['classification_report'])

## 5. Side-by-side comparison

Same test set, same metric definitions, same evaluator code path — so anything that differs in these numbers is the model, not the bookkeeping.

In [ ]:
summary = pd.DataFrame(
    {
        "VADER": [
            vader_metrics['accuracy'],
            vader_metrics['macro_precision'],
            vader_metrics['macro_recall'],
            vader_metrics['macro_f1'],
            vader_metrics['per_class_f1']['negative'],
            vader_metrics['per_class_f1']['neutral'],
            vader_metrics['per_class_f1']['positive'],
        ],
        "FinBERT": [
            finbert_metrics['accuracy'],
            finbert_metrics['macro_precision'],
            finbert_metrics['macro_recall'],
            finbert_metrics['macro_f1'],
            finbert_metrics['per_class_f1']['negative'],
            finbert_metrics['per_class_f1']['neutral'],
            finbert_metrics['per_class_f1']['positive'],
        ],
    },
    index=[
        "Accuracy",
        "Macro precision",
        "Macro recall",
        "Macro F1",
        "F1 (negative)",
        "F1 (neutral)",
        "F1 (positive)",
    ],
)
summary.round(4)

In [ ]:
predictions = build_predictions_frame(
    sentences=test_df['sentence'].tolist(),
    y_true=test_df['label'].to_numpy(),
    vader_pred=vader_pred,
    vader_compound=vader_compound,
    finbert_pred=finbert_pred,
    finbert_confidence=finbert_conf,
)

raw_cm_path, norm_cm_path = plot_confusion_matrices(vader_metrics, finbert_metrics)
f1_path = plot_per_class_f1(vader_metrics, finbert_metrics)
conf_path = plot_finbert_confidence_hist(predictions)
pr_path = plot_per_class_precision_recall(vader_metrics, finbert_metrics)

for p in [raw_cm_path, norm_cm_path, f1_path, conf_path, pr_path]:
    display(Image(filename=str(p)))

In [ ]:
metrics_path = write_metrics_json({"vader": vader_metrics, "finbert": finbert_metrics})
preds_path = write_predictions_csv(predictions)
edges = extract_edge_cases(predictions, n_each=20)
edges_path = write_disagreements_csv(edges)

print(f"metrics  -> {metrics_path}")
print(f"preds    -> {preds_path}")
print(f"edge cs  -> {edges_path}")

## 6. Edge-case deep dive

Numbers are necessary but not sufficient — the interesting question is *which* sentences flip between models. The next cells pull a handful of examples in each interesting category and let us read them.

### Where FinBERT wins (VADER wrong, FinBERT right)

These are the canonical "FinBERT understands financial context" examples. Watch for sentences with words like "compressed," "missed," "tightened," "declined" used in finance-specific senses — VADER's lexicon either has no entry or has the everyday sense, while FinBERT picks up the in-domain meaning.

In [ ]:
finbert_wins = edges[edges['category'] == 'finbert_wins'].head(8)
finbert_wins[['sentence', 'true_label_str', 'vader_pred_str',
               'finbert_pred_str', 'vader_compound', 'finbert_confidence']]

### Where VADER (surprisingly) wins

FinBERT is not magic. The cases where VADER beats it tend to be sentences with strong everyday-sentiment words that happen to align with the gold label, where FinBERT has been thrown off by financial framing that didn't actually flip the polarity.

In [ ]:
vader_wins = edges[edges['category'] == 'vader_wins'].head(5)
vader_wins[['sentence', 'true_label_str', 'vader_pred_str',
             'finbert_pred_str', 'vader_compound', 'finbert_confidence']]

### Where both models fail

Useful for the report — these are the sentences that are genuinely ambiguous, depend on knowledge outside the headline, or are mislabeled in the original dataset (PhraseBank's annotations aren't perfect).

In [ ]:
both_wrong = edges[edges['category'] == 'both_wrong'].head(5)
both_wrong[['sentence', 'true_label_str', 'vader_pred_str',
             'finbert_pred_str', 'vader_compound', 'finbert_confidence']]

### FinBERT's most embarrassing failures

Predictions where FinBERT was very confident and still wrong. The confidence histogram earlier shows the bulk of the wrong predictions are low-confidence (which is what we want), but the long-tail high-confidence errors are worth eyeballing — they tell us where the fine-tune is over-fit or where PhraseBank labels disagree with reasonable readings.

In [ ]:
overconf = edges[edges['category'] == 'finbert_overconfident_wrong'].head(5)
overconf[['sentence', 'true_label_str', 'finbert_pred_str',
           'finbert_confidence']]

## 7. Conclusion

On the same stratified PhraseBank-75agree test set, FinBERT outperforms VADER on accuracy and on macro-F1 by a wide margin, with the largest gap on the **negative** class — exactly where rule-based sentiment is weakest, because financial-negative sentences ("margins compressed," "missed estimates," "trimmed guidance") rarely contain words VADER's lexicon flags as negative. VADER does well on the neutral class, where most sentences are descriptive and have low compound scores by construction.

The bigger pedagogical point is the architectural one. VADER's representation of a sentence is a sum of lexicon look-ups, modulated by a few hand-coded rules. FinBERT's representation is the output of twelve Transformer blocks, each one doing self-attention over every token, with weights fine-tuned on financial text. Same input — "Acme beat estimates by 4 cents" — but VADER sees a bag of words while FinBERT sees a contextually grounded sequence in which "beat" attends to "estimates" and ends up near other positive earnings verbs in vector space. That's the leap from a lexicon to a learned encoder, and it's where the accuracy comes from.

**Caveats and next steps.**

- Both models are zero-shot here. Fine-tuning a small head on the train split on top of FinBERT's encoder would likely push macro-F1 higher; this notebook deliberately keeps the comparison zero-shot so the architecture is what's being measured.
- The PhraseBank labels themselves disagree at the margin (that's literally what the 75% / 50% split is about). The `both_wrong` category above is partly model failure and partly noisy gold.
- The 50%-agreement split is available as an ablation through `load_and_split(use_50agree=True)` if you want to see how the picture changes on a noisier corpus.

All numbers above can be reproduced with `jupyter nbconvert --to notebook --execute notebooks/comparative_analysis.ipynb` from the repo root.